# **Timeseries project**

## **Import libraries**

In [1]:
# Import basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
import keras
import torch
import warnings
warnings.filterwarnings('ignore')

## **Load Data**

In [2]:
import pandas as pd
import os

for split_dataset in ['train', 'test', 'validation']:
    path = f'../data/{split_dataset}'
    print(f"Dataset: {split_dataset}")
    for i in os.listdir(path):
        df = pd.read_csv(f'{path}/{i}', parse_dates=['Date'])
        print(f"{split_dataset}/{i}: {len(df)} rows, "
              f"NaN={df.isnull().sum().sum()}, "
              f"Date range: {df['Date'].min()} -> {df['Date'].max()}")

Dataset: train
train/1.csv: 495 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/10.csv: 494 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/12.csv: 495 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/14.csv: 495 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/16.csv: 495 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/17.csv: 478 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/18.csv: 495 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/2.csv: 402 rows, NaN=0, Date range: 2024-09-23 00:00:00 -> 2026-05-13 00:00:00
train/20.csv: 495 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/21.csv: 189 rows, NaN=0, Date range: 2025-08-04 00:00:00 -> 2026-05-13 00:00:00
train/24.csv: 494 rows, NaN=0, Date range: 2024-05-13 00:00:00 -> 2026-05-13 00:00:00
train/25.csv: 495 rows, NaN=0, Date range

### **Insights**
#### **- Dataset contains 25 individual stocks from the Athens Stock Exchange**
#### **- Split is stock-based (not time-based): 17 train / 3 validation / 5 test stocks**
#### **- Date Range is 2 years from 2024-05-13 to 2026-05-13 (trading days only, Mon-Fri)**
#### **- ~495 trading days per stock (excluding weekends and ASE holidays)**
#### **- No missing values (NaN=0) across all stocks and all splits (train/test/validation)**
#### **- 3 stocks have shorter histories (2.csv, 6.csv, 21.csv) as they are newly listed**


### **We should probably build concatenated df for each split. We should union all stocks of each split together**

#### Load all stock dataframes in a single dictionary (key=split and df is the stock)

In [3]:
all_dataframes = {}

for split_dataset in ['train', 'test', 'validation']:
    path = f'../data/{split_dataset}'
    all_dataframes[split_dataset] = {}  # nested dictionary per split {split: {stock_id: dataframe}}
    
    for i in sorted(os.listdir(path)):
        if i.endswith('.csv'):
            stock_id = i.replace('.csv', '')  #  we split dataframes by stock_id '1.csv' as'1'
            df = pd.read_csv(f'{path}/{i}', parse_dates=['Date'])
            df = df.sort_values('Date').reset_index(drop=True)
            all_dataframes[split_dataset][f'stock_{stock_id}'] = df

# Verify
for split, stocks in all_dataframes.items():
    print(f"{split}: {list(stocks.keys())}")

train: ['stock_1', 'stock_10', 'stock_12', 'stock_14', 'stock_16', 'stock_17', 'stock_18', 'stock_2', 'stock_20', 'stock_21', 'stock_24', 'stock_25', 'stock_3', 'stock_4', 'stock_5', 'stock_6', 'stock_7']
test: ['stock_11', 'stock_13', 'stock_15', 'stock_22', 'stock_8']
validation: ['stock_19', 'stock_23', 'stock_9']


In [4]:
all_dataframes.keys()

dict_keys(['train', 'test', 'validation'])

In [5]:
all_dataframes.values()

dict_values([{'stock_1':           Date  Close   High    Low   Open  Volume
0   2024-05-13   5.43   5.50   5.37   5.49  233043
1   2024-05-14   5.33   5.41   5.33   5.40   37146
2   2024-05-15   5.35   5.36   5.25   5.30  336069
3   2024-05-16   5.29   5.39   5.18   5.39  204403
4   2024-05-17   5.26   5.34   5.20   5.25  162397
..         ...    ...    ...    ...    ...     ...
490 2026-05-07  11.30  11.50  11.14  11.44  192095
491 2026-05-08  11.26  11.32  11.10  11.30  236903
492 2026-05-11  11.20  11.36  11.08  11.16  140573
493 2026-05-12  10.90  11.30  10.86  11.06  204700
494 2026-05-13  10.80  10.98  10.70  10.98   25991

[495 rows x 6 columns], 'stock_10':           Date   Close    High     Low    Open   Volume
0   2024-05-13   8.350   8.406   8.330   8.400   147801
1   2024-05-14   8.310   8.390   8.310   8.350   128421
2   2024-05-15   8.332   8.450   8.332   8.400   322010
3   2024-05-16   8.350   8.410   8.350   8.386   308399
4   2024-05-17   8.356   8.400   8.332   8.400

#### Check first 5 rows of stock 1 dataframe in train split

In [6]:
all_dataframes['train']['stock_1'].head()

,Date,Close,High,Low,Open,Volume
0,2024-05-13,5.43,5.50,5.37,5.49,233043
1,2024-05-14,5.33,5.41,5.33,5.40,37146
2,2024-05-15,5.35,5.36,5.25,5.30,336069
3,2024-05-16,5.29,5.39,5.18,5.39,204403
4,2024-05-17,5.26,5.34,5.20,5.25,162397


#### 

#### Check all Date column's value for stock 1, to figure if only trading days are recording

In [14]:
all_dataframes['train']['stock_1'].columns


Index(['Date', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

In [ ]:
df = all_dataframes['train']['stock_1'].copy()
df['DayOfWeek'] = df['Date'].dt.day_name()
df[df['Date'] > '2024-12-21'][['Date', 'DayOfWeek']].head(20) 
# we choose Christmas period for the analysis to see if there are any events in the data during off days like Christmas and New Year's Day.

,Date,DayOfWeek
157,2024-12-23,Monday
158,2024-12-27,Friday
159,2024-12-30,Monday
160,2024-12-31,Tuesday
161,2025-01-02,Thursday
162,2025-01-03,Friday
163,2025-01-07,Tuesday
164,2025-01-08,Wednesday
165,2025-01-09,Thursday
166,2025-01-10,Friday


#### **Data are only available for trading days (Monday to Friday), excluding weekends and ASE public holidays**

#### Create 5-days training windows in each stock dataframe

In [ ]:
def create_windows(df, input_size=5, output_size=5):
    """
    Input:  Open, High, Low, Close, and Volume data for 5 consecutive trading days (current week)  → shape (5, 5)
    Target: Close prices for next 5 consecutive trading days (next week) → shape (5,)
    """
    X, y = [], []
    
    input_data = df[['Open', 'High', 'Low', 'Close', 'Volume']].values
    close = df['Close'].values
    
    for i in range(len(df) - input_size - output_size + 1):
        X.append(input_data[i : i + input_size])                                   
        y.append(close[i + input_size : i + input_size + output_size])         
    
    return np.array(X), np.array(y)


# Create overlapping windows for all splits
all_windows = {}

for split, stocks in all_dataframes.items():
    X_list, y_list = [], []
    
    for stock_name, df in stocks.items():
        X, y = create_windows(df)
        X_list.append(X)
        y_list.append(y)
    
    all_windows[split] = {
        'X': np.concatenate(X_list, axis=0),  # (total_windows, 5, 5)
        'y': np.concatenate(y_list, axis=0)   # (total_windows, 5)
    }

# Verify
for split, data in all_windows.items():
    print(f"{split}: X={data['X'].shape}, y={data['y'].shape}")


    

train: X=(7561, 5, 5), y=(7561, 5)
test: X=(2430, 5, 5), y=(2430, 5)
validation: X=(1458, 5, 5), y=(1458, 5)


In [34]:

for split in all_dataframes:
    sum=0
    for stock_name, df in all_dataframes[split].items():  # We take each stock dataframe in the split
        # we sum up the number of rows in each dataframe to get the total number of rows persplit
        sum += df.shape[0]
    print(f"{split} total: {sum} rows")

train total: 7714 rows
test total: 2475 rows
validation total: 1485 rows


In [ ]:
if data['X'].shape[0] == data['y'].shape[0]: